In [1]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ============================================================
# CONTROLLERS — change these whenever you want
# ============================================================
CONTEXT_ROW_INDEX = 0
NUMBER_OF_RESULTS = 15000
# ============================================================

PROJECT_ROOT = Path(r"E:\ALL MCP Servers\Recommendation")

MODEL_DIR = PROJECT_ROOT / "models" / "sasrec_basket_v1"

REFRAME_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "chronological_split_70_15_15"
    / "training_reframe"
)

CONTEXT_VECTOR_FILE = (
    MODEL_DIR / "final_sasrec_test_context_vectors.npy"
)

CONTEXT_METADATA_FILE = (
    MODEL_DIR / "final_sasrec_test_context_metadata.csv"
)

FINAL_SASREC_PATH = MODEL_DIR / "best_model_by_ndcg10.pt"

# CSV output directory.
OUTPUT_DIR = MODEL_DIR / "test_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Your SASRec model class must already have been created earlier.
assert "sasrec_model" in globals(), (
    "Run the earlier SASRec model-definition cell first."
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Load the selected final model.
checkpoint = torch.load(
    FINAL_SASREC_PATH,
    map_location=device,
    weights_only=False,
)

sasrec_model.load_state_dict(
    checkpoint["model_state_dict"]
)

sasrec_model.to(device)
sasrec_model.eval()

# Load test-context information and context vectors.
context_metadata = pd.read_csv(
    CONTEXT_METADATA_FILE
)

saved_contexts = np.load(
    CONTEXT_VECTOR_FILE,
    mmap_mode="r",
)

assert len(context_metadata) == len(saved_contexts)
assert 0 <= CONTEXT_ROW_INDEX < len(context_metadata)

# Load index → original Retailrocket item-ID lookup.
item_mapping = pd.read_csv(
    REFRAME_DIR / "train_item_id_mapping.csv"
)

NUM_TRAIN_ITEMS = int(
    item_mapping["item_index"].max()
)

index_to_itemid = np.zeros(
    NUM_TRAIN_ITEMS + 1,
    dtype=np.int64,
)

index_to_itemid[
    item_mapping["item_index"].to_numpy(dtype=np.int64)
] = item_mapping["itemid"].to_numpy(dtype=np.int64)

assert (index_to_itemid[1:] != 0).all()

# Information about the selected test example.
example_information = context_metadata.iloc[
    CONTEXT_ROW_INDEX
]

# Copy the context to avoid read-only NumPy warnings.
context_vector = torch.from_numpy(
    np.asarray(
        saved_contexts[
            CONTEXT_ROW_INDEX:CONTEXT_ROW_INDEX + 1
        ]
    ).copy()
).to(
    device,
    dtype=torch.float32,
)

# Score all training-known catalogue items.
with torch.inference_mode():
    catalogue_item_vectors = (
        sasrec_model.basket_encoder.item_embedding.weight[1:]
    )

    full_scores = torch.matmul(
        context_vector,
        catalogue_item_vectors.T,
    ) * (catalogue_item_vectors.size(1) ** -0.5)

full_scores = full_scores.squeeze(0).float().cpu()

NUMBER_OF_RESULTS = min(
    NUMBER_OF_RESULTS,
    NUM_TRAIN_ITEMS,
)

# Get the highest-scoring predictions.
top_scores, top_positions = torch.topk(
    full_scores,
    k=NUMBER_OF_RESULTS,
)

# +1 because item index 0 is reserved for padding.
top_item_indices = top_positions.numpy() + 1

# Actual next item(s) from the test dataset.
actual_item_indices = json.loads(
    example_information["target_item_indices"]
)

actual_itemids = [
    int(index_to_itemid[item_index])
    for item_index in actual_item_indices
]

# ============================================================
# SUMMARY OUTPUT
# ============================================================
predicted_itemid = int(
    index_to_itemid[top_item_indices[0]]
)

predicted_score = float(top_scores[0])

clean_prediction_output = pd.DataFrame(
    [
        {
            "context_row_index": CONTEXT_ROW_INDEX,
            "visitorid": int(
                example_information["visitorid"]
            ),
            "target_timestamp": int(
                example_information["target_timestamp"]
            ),
            "expected_next_itemids": json.dumps(
                actual_itemids
            ),
            "predicted_next_itemid": predicted_itemid,
            "predicted_logit_score": round(
                predicted_score,
                6,
            ),
            "top_1_prediction_correct": (
                predicted_itemid in actual_itemids
            ),
        }
    ]
)

# ============================================================
# RANKED PREDICTIONS OUTPUT
# ============================================================
ranked_prediction_output = pd.DataFrame(
    {
        "rank": np.arange(
            1,
            NUMBER_OF_RESULTS + 1,
        ),
        "predicted_item_index": top_item_indices,
        "predicted_itemid": index_to_itemid[
            top_item_indices
        ],
        "logit_score": top_scores.numpy().round(6),
    }
)

ranked_prediction_output["is_actual_next_item"] = (
    ranked_prediction_output["predicted_itemid"]
    .isin(actual_itemids)
)

# ============================================================
# SAVE BOTH CSV FILES WITH THE SAME TIMESTAMP
# ============================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

summary_csv_path = (
    OUTPUT_DIR / f"test_summary_{timestamp}.csv"
)

ranked_csv_path = (
    OUTPUT_DIR / f"test_ranked_predictions_{timestamp}.csv"
)

clean_prediction_output.to_csv(
    summary_csv_path,
    index=False,
)

ranked_prediction_output.to_csv(
    ranked_csv_path,
    index=False,
)

# ============================================================
# DISPLAY RESULTS
# ============================================================
print(f"Test example row: {CONTEXT_ROW_INDEX:,}")
print(
    f"Visitor: {int(example_information['visitorid']):,}"
)
print("Expected next item(s):", actual_itemids)
print(f"Showing top {NUMBER_OF_RESULTS:,} recommendations.")

print("\n✅ Summary CSV saved to:")
print(summary_csv_path)

print("\n✅ Ranked predictions CSV saved to:")
print(ranked_csv_path)

display(clean_prediction_output)
display(ranked_prediction_output)

AssertionError: Run the earlier SASRec model-definition cell first.